In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e1/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e1/train.csv
/kaggle/input/competitions/playground-series-s6e1/test.csv


In [2]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e1/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e1/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/playground-series-s6e1/sample_submission.csv"

In [3]:
# A_leitura_normal.py
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Caminhos fornecidos
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e1/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e1/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/playground-series-s6e1/sample_submission.csv"
OUTPUT_PATH = "sample_submission_filled.csv"  # será salvo no workspace do kernel

def safe_read(path, nrows=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    try:
        return pd.read_csv(path, nrows=nrows, sep=None, engine="python")
    except Exception:
        return pd.read_csv(path, nrows=nrows)

# Ler cabeçalhos para detectar colunas
sample_head = safe_read(SAMPLE_SUB_PATH, nrows=0)
train_head = safe_read(TRAIN_PATH, nrows=0)
test_head = safe_read(TEST_PATH, nrows=0)

print("sample cols:", list(sample_head.columns))
print("train cols:", list(train_head.columns))
print("test cols:", list(test_head.columns))

def find_col(df_cols, candidates):
    cols_map = {c.lower(): c for c in df_cols}
    for cand in candidates:
        if cand.lower() in cols_map:
            return cols_map[cand.lower()]
    return None

id_col = find_col(sample_head.columns, ["id", "ID", "Id"])
target_col = find_col(train_head.columns, ["exam_score", "score", "target", "exam score", "examScore"])

if id_col is None:
    raise ValueError("Não foi possível detectar a coluna id no sample_submission.csv")
if target_col is None:
    raise ValueError("Não foi possível detectar a coluna alvo no train.csv")

# Ler dados completos
train = safe_read(TRAIN_PATH)
test = safe_read(TEST_PATH)
sample = safe_read(SAMPLE_SUB_PATH)

# Preparar X e y
X = train.drop(columns=[id_col, target_col], errors="ignore")
y = train[target_col]
X_test = test.drop(columns=[id_col], errors="ignore")

# Processamento simples: priorizar numéricas, senão one-hot
X_num = X.select_dtypes(include=[np.number])
if X_num.shape[1] == 0:
    X_proc = pd.get_dummies(X, drop_first=True)
    X_test_proc = pd.get_dummies(X_test, drop_first=True).reindex(columns=X_proc.columns, fill_value=0)
else:
    X_proc = X_num.fillna(X_num.median())
    X_test_proc = X_test.select_dtypes(include=[np.number]).reindex(columns=X_proc.columns, fill_value=0).fillna(X_proc.median())

# Treino e avaliação
X_tr, X_val, y_tr, y_val = train_test_split(X_proc, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
y_val_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
print(f"RMSE validação: {rmse:.4f}")

# Prever e alinhar por id
preds = model.predict(X_test_proc)
preds_df = pd.DataFrame({id_col: test[id_col].values, "exam_score": preds})

# Alinhar com sample_submission (mantendo ordem do sample)
out = sample.merge(preds_df, on=id_col, how="left")
if "exam_score_y" in out.columns:
    out["exam_score"] = out["exam_score_y"].fillna(out.get("exam_score_x"))
out = out[[c for c in out.columns if not c.endswith("_x") and not c.endswith("_y")]]
out["exam_score"] = out["exam_score"].clip(0, 100)
out.to_csv(OUTPUT_PATH, index=False)
print("Submissão salva em:", OUTPUT_PATH)


sample cols: ['id', 'exam_score']
train cols: ['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty', 'exam_score']
test cols: ['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
RMSE validação: 10.9286
Submissão salva em: sample_submission_filled.csv
